# Market-Oriented Reactive Power Sharing and Voltage Sensitivity Enhancement Operating Data Exploration with `mlcroissant`
This notebook provides a guide for loading and exploring the FAIR^2 dataset using the `mlcroissant` library.

### Dataset Source
The dataset source is provided via a Croissant schema URL:

`https://sen.science/doi/10.71728/senscience.cj0j-ft61/fair2.json`

In [ ]:
# Ensure `mlcroissant` library is installed
!pip install mlcroissant matplotlib seaborn

## 1. Data Loading
Load metadata and records from the dataset using `mlcroissant`.

In [ ]:
import mlcroissant as mlc
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

# Define the dataset URL
url = 'https://sen.science/doi/10.71728/senscience.cj0j-ft61/fair2.json'

import sys
import tempfile
import urllib.request

PATCH_URL = 'https://raw.githubusercontent.com/senscience/notebooks/main/tools/patch_fair2.py'
patch_folder = tempfile.mkdtemp()
urllib.request.urlretrieve(PATCH_URL, f'{patch_folder}/patch_fair2.py')
sys.path.insert(0, patch_folder)
from patch_fair2 import load_dataset

# Load the dataset metadata
dataset = load_dataset(url)

# Access metadata information
metadata = dataset.metadata

print(f"{metadata.name}: {metadata.description}")

## 2. Data Overview
Review available record sets, fields, columns, and their `@id` values using the dataset metadata.

In [ ]:
# The record sets of the dataset, with their fields
record_sets = metadata.record_sets
print(f'Found {len(record_sets)} record sets.')

for record_set in record_sets:
    print(f'--- Record set: {record_set.name} (@id: {record_set.id})')
    print('    Fields:', ', '.join(field.name for field in record_set.fields))

## 3. Data Extraction
Load the records of each record set as a pandas DataFrame, with the field names as columns.

In [ ]:
# Load the records of each record set as a DataFrame, by record set name. mlcroissant names the
# columns by field @id and returns text as bytes: the columns get the field names, text is decoded.
def decode(value):
    return value.decode('utf-8') if isinstance(value, bytes) else value

dataframes = {}
for record_set in record_sets:
    records = list(dataset.records(record_set=record_set.id))
    df = pd.DataFrame(records).rename(columns={field.id: field.name for field in record_set.fields})
    dataframes[record_set.name] = df.apply(lambda column: column.map(decode))
    print(f'{record_set.name}: {df.shape[0]} rows x {df.shape[1]} columns')

## 4. Exploratory Data Analysis (EDA)
Apply common data processing steps to the operating points of the buses of both power systems: filtering the buses with a high reactive power imbalance, normalizing their voltage sensitivity, and grouping them by system and voltage level.

In [ ]:
# The operating point of each bus of both systems, before and after the proposed control and
# market framework: reactive power imbalance, voltage, and voltage sensitivity (dV/dQ)
buses = pd.concat([dataframes['cigre32_bus_operating_points.csv'],
                   dataframes['nordic44_bus_operating_points.csv']], ignore_index=True)
print(f"{len(buses)} buses: {buses['System'].value_counts().to_dict()}")

# Filtering: the buses with a reactive power imbalance above 10% before the framework
high_imbalance = buses[buses['Reactive_Imbalance_Before_pct'] > 10]
print(f'\n{len(high_imbalance)} buses with a reactive power imbalance above 10% before:')
display(high_imbalance[['System', 'Bus_ID', 'Reactive_Imbalance_Before_pct', 'Reactive_Imbalance_After_pct']].head())

# Normalization: the voltage sensitivity as a z-score within each system
buses['Sensitivity_zscore'] = buses.groupby('System')['Sensitivity_dVdQ_pu_per_MVAR'].transform(
    lambda s: (s - s.mean()) / s.std())
display(buses[['System', 'Bus_ID', 'Sensitivity_dVdQ_pu_per_MVAR', 'Sensitivity_zscore']].head())

# Grouping: mean reactive power imbalance and voltage, before and after, by system and voltage level
summary = buses.groupby(['System', 'Voltage_Level_kV'])[[
    'Reactive_Imbalance_Before_pct', 'Reactive_Imbalance_After_pct', 'V_before_pu', 'V_after_pu']].mean()
display(summary.round(3))

## 5. Visualization
Compare the reactive power imbalance and the voltage of each bus before and after the proposed control and market framework.

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(13, 5))

# Reactive power imbalance of each bus, before and after the framework
sns.scatterplot(data=buses, x='Reactive_Imbalance_Before_pct', y='Reactive_Imbalance_After_pct',
                hue='System', ax=axes[0])
limit = buses['Reactive_Imbalance_Before_pct'].max()
axes[0].plot([0, limit], [0, limit], color='grey', linestyle='--', linewidth=1, label='No change')
axes[0].set(title='Reactive power imbalance per bus', xlabel='Before (%)', ylabel='After (%)')
axes[0].legend()

# Bus voltages before and after the framework, by system
voltages = buses.melt(id_vars='System', value_vars=['V_before_pu', 'V_after_pu'],
                      var_name='Framework', value_name='Voltage (pu)')
voltages['Framework'] = voltages['Framework'].map({'V_before_pu': 'Before', 'V_after_pu': 'After'})
sns.boxplot(data=voltages, x='System', y='Voltage (pu)', hue='Framework', ax=axes[1])
axes[1].set(title='Bus voltages before and after')

plt.tight_layout()
plt.show()

### 5.1 Reactive power market and voltage control areas
The proposed framework clears a local reactive power price in each voltage control area (VCA). On the left, the distribution of the clearing price paid at each bus of both systems. On the right, how well the VCAs are partitioned: the internal coupling index (voltage sensitivity between buses of the same VCA) against the external one (sensitivity to buses outside it), from the `vca_market_data.csv` record set.

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Clearing price at each bus, by system (same system colors as the plots above)
sns.histplot(data=buses, x='Clearing_Price_USD_per_MVAR', hue='System', bins=np.arange(20, 80, 5),
             multiple='dodge', shrink=0.85, edgecolor='white', ax=axes[0])
axes[0].set(title='Market clearing price distribution', xlabel='Clearing price (USD/MVAR)',
            ylabel='Number of buses')

# Internal vs external coupling index of each VCA (N44: Nordic 44, C32: CIGRE 32)
vca = dataframes['vca_market_data.csv'].copy()
vca['VCA'] = (vca['System'].map({'Nordic 44': 'N44', 'CIGRE 32': 'C32'})
              + ' VCA' + vca['VCA_ID'].astype(str))
coupling = vca.melt(id_vars=['System', 'VCA'],
                    value_vars=['Internal_Coupling_Index', 'External_Coupling_Index'],
                    var_name='Coupling', value_name='Coupling index')
coupling['Coupling'] = coupling['Coupling'].map({'Internal_Coupling_Index': 'Internal',
                                                 'External_Coupling_Index': 'External'})
sns.barplot(data=coupling, x='VCA', y='Coupling index', hue='Coupling',
            palette={'Internal': '#1B9E77', 'External': '#7570B3'}, ax=axes[1])
axes[1].set(title='VCA coupling indices', xlabel='')
axes[1].tick_params(axis='x', rotation=45)
axes[1].legend(title='Coupling')

plt.tight_layout()
plt.show()

ratio = (vca['Internal_Coupling_Index'] / vca['External_Coupling_Index']).min()
print(f'In every VCA, the internal coupling is at least {ratio:.1f} times the external one.')

## 6. Conclusion
In this notebook, we've loaded and explored the **Market-Oriented Reactive Power Sharing and Voltage Sensitivity Enhancement Operating Data for Nordic 44 and CIGRE 32 Power Systems** dataset using `mlcroissant`.

- The record sets and their fields were reviewed from the Croissant metadata.
- The records of each record set were loaded as DataFrames.
- The operating points of the buses of both systems were filtered, normalized and grouped.
- The reactive power imbalance and the voltages of the buses were compared before and after the proposed framework.
- The reactive power clearing prices of the buses and the coupling of the voltage control areas were compared across both systems.

Further exploration can extend to the other record sets: the hourly cases, the voltage sensitivity matrices, and the cost, performance and stakeholder comparisons.